# BAL Score Changes, Teacher Comparison, and Regional Matched Student Counts for Cambridge TKT-Based Professional Development Program Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This dataset contains teacher-level summaries of matched student English BAL summative assessment marks for teachers in Azerbaijan who participated in an online Cambridge TKT-based professional development programme.

> Important: throughout this notebook, dataset entities are referenced by their Croissant `@id` values whenever record sets, fields, columns, or other schema elements are discussed.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the Croissant URL in a variable so the notebook can be reused easily with a different dataset later.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 120)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.f74k-xed8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name)
print(metadata.description)

### Metadata summary
In this section we inspect high-level metadata fields safely, without treating `dataset.metadata` like a dictionary. When convenient for display, we convert the metadata object to JSON with `.to_json()`.

In [ ]:
metadata_json = metadata.to_json()

summary_keys = [
    '@id', 'name', 'description', 'identifier', 'license', 'version',
    'citeAs', 'datePublished', 'spatialCoverage', 'temporalCoverage'
]

summary_rows = []
for key in summary_keys:
    summary_rows.append({'metadata_field': key, 'value': metadata_json.get(key)})

pd.DataFrame(summary_rows)

## 2. Data Overview
Review available record sets, fields, and their IDs.

For this dataset, the provided Croissant package indicates that `recordSet` is empty, so there may be no tabular record sets directly exposed through the schema. The notebook therefore first checks what is available and then falls back to a metadata-driven exploration if needed.

In [ ]:
record_sets_json = metadata_json.get('recordSet', [])
print(f'Number of record sets declared in metadata: {len(record_sets_json)}')
record_sets_json

### Inspect record set declarations by `@id`
If record sets exist, this cell lists their `@id`, names, and fields. Otherwise, it confirms that the dataset is metadata-only from the perspective of Croissant records.

In [ ]:
if record_sets_json:
    overview_rows = []
    for rs in record_sets_json:
        rs_id = rs.get('@id')
        rs_name = rs.get('name')
        rs_fields = rs.get('field', [])
        overview_rows.append({
            'record_set_@id': rs_id,
            'record_set_name': rs_name,
            'n_fields': len(rs_fields)
        })
    display(pd.DataFrame(overview_rows))
else:
    print('No Croissant record sets are declared in metadata_json[\'recordSet\'].')
    print('This means dataset.records(record_set=<@id>) is not expected to yield tabular rows for this package.')

### Attempt to iterate over records using a record set `@id`
The template includes an example of iterating over records in a record set. Here we do that conditionally, using the first available record set `@id` if one exists.

In [ ]:
if record_sets_json:
    example_record_set_id = record_sets_json[0].get('@id')
    print(f'Using record set @id: {example_record_set_id}')
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i >= 4:
            break
else:
    print('No record set @id is available, so record iteration is skipped.')

### Inspect distributions and other dataset entities by `@id`
Even when no record sets are defined, the metadata still contains useful dataset entities such as distributions. We inspect those entities using their `@id` values.

In [ ]:
distribution_json = metadata_json.get('distribution', [])

distribution_rows = []
for d in distribution_json:
    distribution_rows.append({
        'distribution_@id': d.get('@id'),
        'name': d.get('name'),
        'contentUrl': d.get('contentUrl'),
        'encodingFormat': d.get('encodingFormat')
    })

pd.DataFrame(distribution_rows) if distribution_rows else pd.DataFrame(columns=['distribution_@id', 'name', 'contentUrl', 'encodingFormat'])

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis when record sets are available.

Because this dataset appears to expose no Croissant `recordSet` entities, we create analysis-ready DataFrames from structured metadata instead. This keeps the notebook useful and aligned with the schema as published.

In [ ]:
# Extract data from each record set if available
record_set_ids = [rs.get('@id') for rs in record_sets_json if rs.get('@id')]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No record-set DataFrames were created because no record sets were defined.')

### Build metadata-based DataFrames
These tables are derived from the Croissant metadata JSON and are useful for exploration even without row-level record sets.

In [ ]:
# 1) Dataset-level scalar and list metadata flattened into a long table
interesting_fields = [
    '@id', 'name', 'identifier', 'version', 'datePublished', 'license',
    'spatialCoverage', 'temporalCoverage', 'citeAs',
    'dataCollection', 'dataCollectionMissingData', 'dataCollectionRawData',
    'dataManipulationProtocol', 'dataSocialImpact'
]

dataset_overview_df = pd.DataFrame([
    {'field_@id_or_name': field_name, 'value': metadata_json.get(field_name)}
    for field_name in interesting_fields
])

# 2) List-based analytical tables
data_biases_df = pd.DataFrame({'dataBiases_item': metadata_json.get('dataBiases', [])})
data_limitations_df = pd.DataFrame({'dataLimitations_item': metadata_json.get('dataLimitations', [])})
data_use_cases_df = pd.DataFrame({'dataUseCases_item': metadata_json.get('dataUseCases', [])})
data_collection_timeframe_df = pd.DataFrame({'dataCollectionTimeframe_item': metadata_json.get('dataCollectionTimeframe', [])})
keywords_df = pd.DataFrame({'keyword': metadata_json.get('keywords', [])})

dataframes['metadata:dataset_overview'] = dataset_overview_df
dataframes['metadata:dataBiases'] = data_biases_df
dataframes['metadata:dataLimitations'] = data_limitations_df
dataframes['metadata:dataUseCases'] = data_use_cases_df
dataframes['metadata:dataCollectionTimeframe'] = data_collection_timeframe_df
dataframes['metadata:keywords'] = keywords_df

print('Created metadata-based DataFrames:')
print(list(dataframes.keys()))

In [ ]:
dataframes['metadata:dataset_overview'].head(10)

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps to the available data.

Since no row-level record set is exposed, this EDA focuses on:
- counting and organizing metadata elements,
- creating derived variables from the metadata,
- normalizing simple numeric summaries,
- grouping and categorizing text-based metadata.

This is still useful for understanding dataset scope, limitations, and intended use.

In [ ]:
# Build a compact numeric summary table from metadata
numeric_summary_rows = [
    {'metric_@id_or_name': 'n_record_sets', 'value': len(metadata_json.get('recordSet', []))},
    {'metric_@id_or_name': 'n_distributions', 'value': len(metadata_json.get('distribution', []))},
    {'metric_@id_or_name': 'n_keywords', 'value': len(metadata_json.get('keywords', []))},
    {'metric_@id_or_name': 'n_dataBiases', 'value': len(metadata_json.get('dataBiases', []))},
    {'metric_@id_or_name': 'n_dataLimitations', 'value': len(metadata_json.get('dataLimitations', []))},
    {'metric_@id_or_name': 'n_dataUseCases', 'value': len(metadata_json.get('dataUseCases', []))},
    {'metric_@id_or_name': 'n_dataCollectionTimeframe_points', 'value': len(metadata_json.get('dataCollectionTimeframe', []))}
]

eda_metrics_df = pd.DataFrame(numeric_summary_rows)
dataframes['metadata:eda_metrics'] = eda_metrics_df
eda_metrics_df

In [ ]:
# Select a numeric field for analysis
record_set_id = 'metadata:eda_metrics'
numeric_field = 'value'

threshold = 1
filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold].copy()
print(f'Filtered records with {numeric_field} > {threshold}:')
display(filtered_df.head())

std_value = filtered_df[numeric_field].std()
if pd.notnull(std_value) and std_value != 0:
    filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
else:
    filtered_df[f'{numeric_field}_normalized'] = 0.0

print(f'Normalized {numeric_field} for filtered records:')
display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

group_field = 'metric_category'
filtered_df[group_field] = filtered_df['metric_@id_or_name'].str.replace('n_', '', regex=False).str.split('_').str[0]

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
    print(f'Grouped data by {group_field}:')
    display(grouped_df.head())

### Text feature engineering on dataset limitations
We can also derive useful exploratory variables from text fields. Here we measure the length of each limitation statement and classify it into simple size categories.

In [ ]:
limitations_df = dataframes['metadata:dataLimitations'].copy()
limitations_df['text_length'] = limitations_df['dataLimitations_item'].fillna('').str.len()
limitations_df['word_count'] = limitations_df['dataLimitations_item'].fillna('').str.split().str.len()

def length_category(n_words):
    if n_words < 10:
        return 'short'
    elif n_words < 20:
        return 'medium'
    return 'long'

limitations_df['length_category'] = limitations_df['word_count'].apply(length_category)
dataframes['metadata:dataLimitations_enriched'] = limitations_df
limitations_df

### Parse the collection timeframe into dates
The `dataCollectionTimeframe` metadata can be turned into a proper time series for plotting and chronological inspection.

In [ ]:
timeframe_df = dataframes['metadata:dataCollectionTimeframe'].copy()
timeframe_df['date'] = pd.to_datetime(timeframe_df['dataCollectionTimeframe_item'], format='%Y-%m', errors='coerce')
timeframe_df = timeframe_df.sort_values('date').reset_index(drop=True)
timeframe_df['sequence'] = timeframe_df.index + 1
dataframes['metadata:dataCollectionTimeframe_parsed'] = timeframe_df
timeframe_df

## 5. Visualization
Visualize metadata distributions and derived analytical summaries.

Because there are no exposed record sets, the plots below focus on the dataset metadata and its derived numeric/text summaries.

In [ ]:
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: numeric metadata metrics
sns.barplot(
    data=dataframes['metadata:eda_metrics'],
    x='value',
    y='metric_@id_or_name',
    palette='Blues_d',
    ax=axes[0, 0]
)
axes[0, 0].set_title('Dataset-level metadata counts')
axes[0, 0].set_xlabel('Count')
axes[0, 0].set_ylabel('Metric @id or name')

# Plot 2: limitation statement word counts
sns.barplot(
    data=dataframes['metadata:dataLimitations_enriched'],
    x='word_count',
    y=dataframes['metadata:dataLimitations_enriched'].index.astype(str),
    palette='Oranges_d',
    ax=axes[0, 1]
)
axes[0, 1].set_title('Word counts for data limitation statements')
axes[0, 1].set_xlabel('Word count')
axes[0, 1].set_ylabel('Limitation index')

# Plot 3: limitation category counts
length_counts = dataframes['metadata:dataLimitations_enriched']['length_category'].value_counts().reset_index()
length_counts.columns = ['length_category', 'count']
sns.barplot(
    data=length_counts,
    x='length_category',
    y='count',
    palette='Greens_d',
    ax=axes[1, 0]
)
axes[1, 0].set_title('Limitation statement length categories')
axes[1, 0].set_xlabel('Length category')
axes[1, 0].set_ylabel('Count')

# Plot 4: collection timeframe sequence
sns.lineplot(
    data=dataframes['metadata:dataCollectionTimeframe_parsed'],
    x='date',
    y='sequence',
    marker='o',
    ax=axes[1, 1]
)
axes[1, 1].set_title('Data collection timeline points')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Sequence')

plt.tight_layout()
plt.show()

## 6. Conclusion
This notebook demonstrated how to explore a Croissant dataset with `mlcroissant`, while respecting the schema structure and using entity identifiers consistently.

### Key observations
- The dataset metadata loaded successfully from the Croissant URL.
- The package currently exposes **no Croissant `recordSet` entities**, so row-level extraction via `dataset.records(record_set=<@id>)` is not available.
- The metadata itself is still rich and analytically useful, especially for understanding:
  - dataset scope,
  - limitations and biases,
  - intended use cases,
  - distributions and temporal coverage.
- Derived metadata tables can support lightweight EDA even in metadata-only datasets.

### Practical next step
If a future version of this Croissant package adds record sets and fields with explicit `@id` values, the same notebook structure can be reused directly to load those rows into DataFrames for deeper analysis.